# M2 gate-free 저차원 체크포인트 진단

기존 seed 42 M1·M2 체크포인트만 불러옵니다. **재학습은 하지 않습니다.**

출력: (1) 저·중·고CLV별 정답상품 순위 이동, (2) ID/N/V 블록별 성과, (3) 실제 후보점수에서 ID/N/V 축의 표준편차와 ID 대비 비율.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'e2cb5306593e757db01465c8c7f0712cb6d4d52f'
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q {REVIEWED_SHA}

import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('진단 코드 SHA 확인:', REVIEWED_SHA)

In [ ]:
import json
from lightgcn_clv_gatefree_lowdim_diagnostic import (
    configure_checkpoint_diagnostic,
    preflight_summary,
    run_checkpoint_diagnostic,
)

cfg = configure_checkpoint_diagnostic(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_gatefree_lowdim_historical_screen_v1'
    ),
    baseline_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    ),
    eval_batch_size=32,
)
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
# 기존 체크포인트 평가만 수행합니다. optimizer·epoch 학습은 없습니다.
report = run_checkpoint_diagnostic(cfg)

In [ ]:
from IPython.display import display

print('1) 저·중·고CLV별 정답상품 순위 이동')
display(report['rank_transition'])

print('2) ID/N/V 블록별 성과')
display(report['view_metrics'])

print('3) 실제 후보점수 영향력')
display(report['score_strength'])

print('저장 파일')
print(json.dumps(report['paths'], ensure_ascii=False, indent=2))